## **Combinación de predicciones**

En este notebook se proponen diferentes alternativas para combinar los resultados obtenidos por el modelo sobre datos tabulares, el modelo de imágenes y el modelo de texto.

#### **Librerias a utilizar**

In [ ]:
import pandas as pd

#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle

#Visualizacióon
from plotly import express as px

#Plot de matriz de confusion normalizada en actuals
from utils import plot_confusion_matrix

#### **Resultados LightGBM**

In [ ]:
pred_lightgbm = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_lightgbm.csv")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(pred_lightgbm["AdoptionSpeed"], pred_lightgbm["PredDT"]))

cohen_kappa_score(pred_lightgbm["AdoptionSpeed"], pred_lightgbm["PredDT"], weights = 'quadratic')

#### **Resultados imágenes (ResNet50)**

In [ ]:
pred_imagenes = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_imagenes.csv")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(pred_imagenes["AdoptionSpeed"], pred_imagenes["Im_Pred"]))

cohen_kappa_score(pred_imagenes["AdoptionSpeed"], pred_imagenes["Im_Pred"], weights = 'quadratic')

#### **Resultados texto (DistilBERT)**

In [ ]:
pred_txt = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_txt_base.csv")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(pred_txt["AdoptionSpeed"], pred_txt["txt_Pred"]))

cohen_kappa_score(pred_txt["AdoptionSpeed"], pred_txt["txt_Pred"], weights = 'quadratic')

#### **Combinación de Resultados**

In [ ]:
# Combinación de resultados
Resultados = pd.merge(pred_lightgbm, pred_imagenes, on = "PetID", how="left")
Resultados = pd.merge(Resultados, pred_txt, on = "PetID", how="left")

# Se reemplazan los NA de probabilidades por 0
Resultados.loc[:, ~Resultados.columns.isin(["Im_pred", "txt_Pred"])] = Resultados.loc[:, ~Resultados.columns.isin(["Im_pred", "txt_Pred"])].fillna(0)


##### **Promedio de probabilidades**

In [ ]:
# Calcular los promedios y crear 5 nuevas columnas
for i in range(5):
    Resultados[f'Promedio_Clase_{i}'] = (Resultados[f'DT_Clase_{i}'] + Resultados[f'Im_Clase_{i}'] + Resultados[f'txt_Clase_{i}']) / 3

# Realizar la prediccion con las 5 columnas promedio
columnas_promedio = [f'Promedio_Clase_{i}' for i in range(5)]

Resultados['Pred_Promedio'] = Resultados[columnas_promedio].idxmax(axis=1)

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(Resultados["AdoptionSpeed"], Resultados["Pred_Promedio"]))

cohen_kappa_score(Resultados["AdoptionSpeed"], Resultados["Pred_Promedio"], weights = 'quadratic')

##### **Búsqueda de la mejor combinación**

In [ ]:
import itertools

# Valores permitidos
valores = [round(i * 0.05, 1) for i in range(0, 21)]  

# Generar todas las combinaciones con repetición de 3 valores
combinaciones = [
    comb for comb in itertools.product(valores, repeat=3)
    if round(sum(comb), 1) == 1.0
]

max_kappa = 0

for c in combinaciones:
    # Calcular la combinación y crear 5 nuevas columnas
    for i in range(5):
        Resultados[f'Combinacion_Clase_{i}'] = (Resultados[f'DT_Clase_{i}'] * c[0] + Resultados[f'Im_Clase_{i}'] * c[1] + Resultados[f'txt_Clase_{i}'] * c[2])

    # Realizar la prediccion con las 5 columnas combinadas
    columnas_combinacion = [f'Combinacion_Clase_{i}' for i in range(5)]

    Resultados['Pred_Combinacion'] = Resultados[columnas_combinacion].idxmax(axis=1)

    #Calculo matriz de confusion y kappa
    matriz = plot_confusion_matrix(Resultados["AdoptionSpeed"], Resultados["Pred_Combinacion"])

    kappa = cohen_kappa_score(Resultados["AdoptionSpeed"], Resultados["Pred_Combinacion"], weights = 'quadratic')

    if kappa > max_kappa:
        max_kappa = kappa
        max_matrix = matriz
        max_comb = c

display(max_matrix)
print(f'La mejor combinación es {max_comb} con un kappa de {max_kappa}')

Cantidad de combinaciones: 66
(0.0, 0.0, 1.0)
(0.0, 0.1, 0.9)
(0.0, 0.2, 0.8)
(0.0, 0.3, 0.7)
(0.0, 0.4, 0.6)
(0.0, 0.5, 0.5)
(0.0, 0.6, 0.4)
(0.0, 0.7, 0.3)
(0.0, 0.8, 0.2)
(0.0, 0.9, 0.1)
(0.0, 1.0, 0.0)
(0.1, 0.0, 0.9)
(0.1, 0.1, 0.8)
(0.1, 0.2, 0.7)
(0.1, 0.3, 0.6)
(0.1, 0.4, 0.5)
(0.1, 0.5, 0.4)
(0.1, 0.6, 0.3)
(0.1, 0.7, 0.2)
(0.1, 0.8, 0.1)
(0.1, 0.9, 0.0)
(0.2, 0.0, 0.8)
(0.2, 0.1, 0.7)
(0.2, 0.2, 0.6)
(0.2, 0.3, 0.5)
(0.2, 0.4, 0.4)
(0.2, 0.5, 0.3)
(0.2, 0.6, 0.2)
(0.2, 0.7, 0.1)
(0.2, 0.8, 0.0)
(0.3, 0.0, 0.7)
(0.3, 0.1, 0.6)
(0.3, 0.2, 0.5)
(0.3, 0.3, 0.4)
(0.3, 0.4, 0.3)
(0.3, 0.5, 0.2)
(0.3, 0.6, 0.1)
(0.3, 0.7, 0.0)
(0.4, 0.0, 0.6)
(0.4, 0.1, 0.5)
(0.4, 0.2, 0.4)
(0.4, 0.3, 0.3)
(0.4, 0.4, 0.2)
(0.4, 0.5, 0.1)
(0.4, 0.6, 0.0)
(0.5, 0.0, 0.5)
(0.5, 0.1, 0.4)
(0.5, 0.2, 0.3)
(0.5, 0.3, 0.2)
(0.5, 0.4, 0.1)
(0.5, 0.5, 0.0)
(0.6, 0.0, 0.4)
(0.6, 0.1, 0.3)
(0.6, 0.2, 0.2)
(0.6, 0.3, 0.1)
(0.6, 0.4, 0.0)
(0.7, 0.0, 0.3)
(0.7, 0.1, 0.2)
(0.7, 0.2, 0.1)
(0.7, 0.3, 0.0)
(0.8, 0.0,

1.0